# Paper QA With Section Attributions

Given a paper id and a user question, this notebook retrieves paper-scoped KG evidence from Fuseki and returns:

`question`, `answer`, and `attribute_uris`.

The retrieval path reuses the URI resolver's Fuseki helpers and the existing Jena text index. Section URIs are used as the primary attribution target.

## Setup

This expects the repo `.env` to contain `FUSEKI_SERVER_URL` and `FUSEKI_DATASET`. A local paper id such as `learning_relation_entailment_with_structured_and_textual_information` is expanded to `https://w3id.org/twc/sudo/kg/paper/{paper_id}` unless you pass a full URI.

In [3]:
from __future__ import annotations

import json
import logging
import math
import os
import re
from typing import Any
from urllib.parse import quote, urlsplit
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

logging.disable(logging.INFO)
try:
    from uri_resolver.backend import FusekiRedirectBackend
    from uri_resolver.main import (
        _fetch_select_bindings_from_datasets,
        _ordered_dataset_names,
        fetch_fuseki_dataset_names,
    )
    from uri_resolver.settings import AppSettings
except ModuleNotFoundError:
    class AppSettings:
        def __init__(self, _env_file: str | None = None) -> None:
            self.fuseki_server_url = os.environ["FUSEKI_SERVER_URL"]
            self.fuseki_dataset = os.getenv("FUSEKI_DATASET", "gold_standard_kg")
            self.persistent_uri_base = os.getenv(
                "PERSISTENT_URI_BASE",
                "https://w3id.org/twc/sudo/kg",
            )


    class FusekiRedirectBackend:
        def __init__(self, server_url: str, dataset: str = "gold_standard_kg") -> None:
            self.server_url = server_url.rstrip("/")
            self.default_dataset = dataset.strip().strip("/")

        def get_select_target(self, resource_label: str, query: str, dataset: str | None = None) -> str:
            dataset_name = (dataset or self.default_dataset).strip().strip("/")
            return (
                f"{self.server_url}/{dataset_name}/query"
                f"?query={quote(query, safe='')}&output=application%2Fsparql-results%2Bjson"
            )


    def _extract_dataset_names(payload: object) -> list[str]:
        raw_datasets = payload
        if isinstance(payload, dict):
            for key in ("datasets", "dataset", "services"):
                if key in payload:
                    raw_datasets = payload[key]
                    break

        names: list[str] = []
        if isinstance(raw_datasets, list):
            for item in raw_datasets:
                if isinstance(item, str):
                    names.append(item)
                elif isinstance(item, dict):
                    for field in ("name", "dbName", "dataset", "id", "ds.name"):
                        value = item.get(field)
                        if isinstance(value, str) and value.strip():
                            names.append(value)
                            break
        elif isinstance(raw_datasets, dict):
            for key, value in raw_datasets.items():
                if isinstance(key, str) and key.strip():
                    names.append(key)
                if isinstance(value, dict):
                    for field in ("name", "dbName", "dataset", "id", "ds.name"):
                        raw_name = value.get(field)
                        if isinstance(raw_name, str) and raw_name.strip():
                            names.append(raw_name)
                            break

        ordered: list[str] = []
        seen: set[str] = set()
        for name in names:
            normalized = name.strip().strip("/")
            if normalized and normalized not in seen:
                seen.add(normalized)
                ordered.append(normalized)
        return ordered


    def fetch_fuseki_dataset_names(server_url: str, timeout: float = 10.0) -> list[str]:
        for suffix in ("/$/datasets", "/$/server"):
            try:
                request = Request(
                    f"{server_url.rstrip('/')}{suffix}",
                    headers={"Accept": "application/json"},
                )
                with urlopen(request, timeout=timeout) as response:
                    return _extract_dataset_names(json.loads(response.read().decode("utf-8")))
            except Exception:
                continue
        return []


    def _ordered_dataset_names(default_dataset: str, discovered_datasets: list[str]) -> list[str]:
        ordered: list[str] = []
        seen: set[str] = set()
        for dataset_name in [default_dataset, *discovered_datasets]:
            normalized = dataset_name.strip().strip("/")
            if normalized and normalized not in seen:
                seen.add(normalized)
                ordered.append(normalized)
        return ordered


    def fetch_sparql_bindings(url: str, timeout: float = 10.0) -> list[dict[str, object]]:
        request = Request(url, headers={"Accept": "application/sparql-results+json"})
        with urlopen(request, timeout=timeout) as response:
            body = json.loads(response.read().decode("utf-8"))
        bindings = body.get("results", {}).get("bindings", [])
        return bindings if isinstance(bindings, list) else []


    def _fetch_select_bindings_from_datasets(
        backend: FusekiRedirectBackend,
        resource_label: str,
        query: str,
        dataset_names: list[str],
    ) -> list[dict[str, object]]:
        collected: list[dict[str, object]] = []
        seen: set[str] = set()
        for dataset_name in dataset_names:
            try:
                rows = fetch_sparql_bindings(backend.get_select_target(resource_label, query, dataset_name))
            except Exception:
                continue
            for row in rows:
                key = json.dumps(row, sort_keys=True)
                if key not in seen:
                    seen.add(key)
                    collected.append(row)
        return collected
finally:
    logging.disable(logging.NOTSET)


def find_env_file(start: str | None = None) -> str:
    candidate = os.path.abspath(start or os.getenv("ENV_FILE", ".env"))
    if os.path.exists(candidate):
        return candidate
    current = os.path.abspath(os.getcwd())
    while True:
        env_path = os.path.join(current, ".env")
        if os.path.exists(env_path):
            return env_path
        parent = os.path.dirname(current)
        if parent == current:
            return candidate
        current = parent


ENV_FILE = find_env_file()


def load_env_file(path: str = ENV_FILE, *, override: bool = False) -> None:
    """Load simple KEY=VALUE lines from .env without adding a dependency."""
    try:
        lines = open(path, encoding="utf-8").read().splitlines()
    except OSError:
        return

    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or "=" not in stripped:
            continue
        key, raw_value = stripped.split("=", 1)
        key = key.strip()
        value = raw_value.strip().strip('"').strip("'")
        if override or key not in os.environ:
            os.environ[key] = value


load_env_file()


In [4]:
settings = AppSettings(_env_file=ENV_FILE)
logging.getLogger("uri_resolver").setLevel(logging.WARNING)
logging.getLogger("uri_resolver.doc").setLevel(logging.WARNING)
logging.getLogger("uri_resolver.fuseki").setLevel(logging.WARNING)

backend = FusekiRedirectBackend(
    server_url=settings.fuseki_server_url,
    dataset=settings.fuseki_dataset,
)

dataset_names = _ordered_dataset_names(
    settings.fuseki_dataset,
    fetch_fuseki_dataset_names(settings.fuseki_server_url),
)
persistent_uri_base = os.getenv(
    "PAPER_QA_PERSISTENT_URI_BASE",
    settings.persistent_uri_base,
).rstrip("/")

{
    "fuseki_server_url": settings.fuseki_server_url,
    "default_dataset": backend.default_dataset,
    "dataset_search_order": dataset_names,
    "persistent_uri_base": persistent_uri_base,
}


{'fuseki_server_url': 'https://spark-6d47.tailb1f37b.ts.net/fuseki/',
 'default_dataset': 'efemo_kg',
 'dataset_search_order': ['efemo_kg'],
 'persistent_uri_base': 'https://w3id.org/twc/sudo/kg'}

## Retrieval Helpers

The KG stores provenance, labels, and structure across named graphs, so the queries intentionally join across multiple `GRAPH ?g { ... }` blocks instead of requiring all triples to live in one named graph.

In [5]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "how", "in",
    "into", "is", "it", "of", "on", "or", "paper", "that", "the", "their", "this",
    "to", "was", "were", "what", "when", "where", "which", "who", "why", "with",
}


def paper_uri_from_id(paper_id: str) -> str:
    value = paper_id.strip()
    if not value:
        raise ValueError("paper_id must be non-empty")
    if value.startswith("http://") or value.startswith("https://"):
        return value
    return f"{persistent_uri_base}/paper/{quote(value.strip('/'), safe='-._~')}"


def sparql_iri(uri: str) -> str:
    parsed = urlsplit(uri)
    if parsed.scheme not in {"http", "https", "urn"}:
        raise ValueError(f"Unsupported URI for SPARQL IRI: {uri}")
    if any(char in uri for char in "<>\n\r"):
        raise ValueError(f"Unsafe URI for SPARQL IRI: {uri}")
    return f"<{uri}>"


def sparql_string(value: str) -> str:
    return json.dumps(value)


def binding_value(binding: dict[str, Any], name: str) -> str | None:
    cell = binding.get(name)
    if not isinstance(cell, dict):
        return None
    value = cell.get("value")
    return value if isinstance(value, str) else None


def binding_float(binding: dict[str, Any], name: str) -> float | None:
    value = binding_value(binding, name)
    if value is None:
        return None
    try:
        return float(value)
    except ValueError:
        return None


def binding_int(binding: dict[str, Any], name: str) -> int | None:
    value = binding_value(binding, name)
    if value is None:
        return None
    try:
        return int(value)
    except ValueError:
        return None


def question_terms(question: str) -> list[str]:
    terms = re.findall(r"[A-Za-z0-9][A-Za-z0-9_-]+", question.lower())
    return [term for term in terms if len(term) > 2 and term not in STOPWORDS]


def lucene_query_from_question(question: str, max_terms: int = 18) -> str:
    terms = question_terms(question)
    if not terms:
        terms = re.findall(r"[A-Za-z0-9]+", question.lower())
    return " ".join(terms[:max_terms])


def run_select(query: str) -> list[dict[str, Any]]:
    return _fetch_select_bindings_from_datasets(
        backend=backend,
        resource_label="paper_qa",
        query=query,
        dataset_names=dataset_names,
    )


In [6]:
POSITION = "https://w3id.org/twc/sudo/kg/position"


def paper_metadata_query(paper_uri: str) -> str:
    paper = sparql_iri(paper_uri)
    return f"""
PREFIX po: <http://www.essepuntato.it/2008/12/pattern#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?predicate ?value ?displayValue WHERE {{
  GRAPH ?metadataGraph {{
    {paper} ?predicate ?value .
    FILTER(?predicate != po:contains)
  }}
  OPTIONAL {{ GRAPH ?valueLabelGraph {{ ?value rdfs:label ?valueLabel }} }}
  BIND(COALESCE(?valueLabel, STR(?value)) AS ?displayValue)
}}
ORDER BY STR(?predicate) STR(?displayValue)
"""


def relevant_chunks_query(paper_uri: str, question: str, limit: int) -> str:
    paper = sparql_iri(paper_uri)
    query = sparql_string(lucene_query_from_question(question))
    limit = max(1, int(limit))
    return f"""
PREFIX text: <http://jena.apache.org/text#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX po: <http://www.essepuntato.it/2008/12/pattern#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?node ?score ?text ?nodeType ?sentence ?section ?sectionTitle ?sectionPosition ?sentencePosition WHERE {{
  GRAPH ?textGraph {{
    (?node ?score) text:query (rdfs:label {query} {limit}) .
    ?node rdfs:label ?text .
  }}
  GRAPH ?provGraph {{
    ?node prov:hadPrimarySource {paper} .
    OPTIONAL {{ ?node prov:wasDerivedFrom ?sentence }}
  }}
  OPTIONAL {{ GRAPH ?typeGraph {{ ?node rdf:type ?nodeType }} }}
  OPTIONAL {{
    GRAPH ?structureGraph {{
      {paper} po:contains ?section .
      ?section po:contains ?paragraph .
      ?paragraph po:contains ?sentence .
      OPTIONAL {{ ?section <{POSITION}> ?sectionPosition }}
      OPTIONAL {{ ?sentence <{POSITION}> ?sentencePosition }}
      OPTIONAL {{ ?section po:containsAsHeader ?header }}
    }}
    OPTIONAL {{ GRAPH ?headerLabelGraph {{ ?header rdfs:label ?sectionTitleLabel }} }}
    OPTIONAL {{ GRAPH ?headerValueGraph {{ ?header rdf:value ?sectionTitleValue }} }}
    BIND(COALESCE(?sectionTitleLabel, ?sectionTitleValue, STRAFTER(STR(?section), "/section/")) AS ?sectionTitle)
  }}
}}
ORDER BY DESC(?score) ?sectionPosition ?sentencePosition STR(?node)
LIMIT {limit}
"""


def all_paper_chunks_query(paper_uri: str) -> str:
    paper = sparql_iri(paper_uri)
    return f"""
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX po: <http://www.essepuntato.it/2008/12/pattern#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?node ?text ?nodeType ?sentence ?section ?sectionTitle ?sectionPosition ?sentencePosition WHERE {{
  GRAPH ?provGraph {{
    ?node prov:hadPrimarySource {paper} .
    OPTIONAL {{ ?node prov:wasDerivedFrom ?sentence }}
  }}
  GRAPH ?labelGraph {{
    ?node rdfs:label ?text .
  }}
  OPTIONAL {{ GRAPH ?typeGraph {{ ?node rdf:type ?nodeType }} }}
  OPTIONAL {{
    GRAPH ?structureGraph {{
      {paper} po:contains ?section .
      ?section po:contains ?paragraph .
      ?paragraph po:contains ?sentence .
      OPTIONAL {{ ?section <{POSITION}> ?sectionPosition }}
      OPTIONAL {{ ?sentence <{POSITION}> ?sentencePosition }}
      OPTIONAL {{ ?section po:containsAsHeader ?header }}
    }}
    OPTIONAL {{ GRAPH ?headerLabelGraph {{ ?header rdfs:label ?sectionTitleLabel }} }}
    OPTIONAL {{ GRAPH ?headerValueGraph {{ ?header rdf:value ?sectionTitleValue }} }}
    BIND(COALESCE(?sectionTitleLabel, ?sectionTitleValue, STRAFTER(STR(?section), "/section/")) AS ?sectionTitle)
  }}
}}
ORDER BY ?sectionPosition ?sentencePosition STR(?node)
"""


In [7]:
def rows_to_chunks(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    chunks_by_key: dict[tuple[str | None, str | None, str | None, str | None], dict[str, Any]] = {}
    for row in rows:
        node_uri = binding_value(row, "node")
        text = binding_value(row, "text")
        sentence_uri = binding_value(row, "sentence")
        section_uri = binding_value(row, "section")
        if not node_uri or not text:
            continue
        key = (node_uri, text, sentence_uri, section_uri)
        chunk = chunks_by_key.setdefault(
            key,
            {
                "node_uri": node_uri,
                "text": " ".join(text.split()),
                "sentence_uri": sentence_uri,
                "section_uri": section_uri,
                "section_title": binding_value(row, "sectionTitle"),
                "section_position": binding_int(row, "sectionPosition"),
                "sentence_position": binding_int(row, "sentencePosition"),
                "score": binding_float(row, "score"),
                "node_types": [],
            },
        )
        score = binding_float(row, "score")
        if score is not None:
            chunk["score"] = max(score, chunk["score"] or -math.inf)
        node_type = binding_value(row, "nodeType")
        if node_type and node_type not in chunk["node_types"]:
            chunk["node_types"].append(node_type)

    return list(chunks_by_key.values())


def lexical_score(question: str, text: str) -> float:
    terms = question_terms(question)
    words = re.findall(r"[A-Za-z0-9][A-Za-z0-9_-]+", text.lower())
    if not terms or not words:
        return 0.0

    word_set = set(words)
    matched_terms = set()
    overlap = 0.0
    for term in terms:
        prefix = term[:5]
        for word in word_set:
            if word == term or (len(prefix) >= 4 and word.startswith(prefix)):
                matched_terms.add(term)
                overlap += 1.0
                break

    coverage = len(matched_terms) / len(set(terms))
    return overlap + coverage


def chunk_answer_score(question: str, chunk: dict[str, Any]) -> float:
    text = chunk["text"]
    word_count = len(text.split())
    base = lexical_score(question, text)
    lucene = float(chunk.get("score") or 0.0)
    uri = str(chunk.get("node_uri") or "")
    substantive_bonus = 2.0 if word_count >= 8 else -1.0
    proposition_bonus = 1.5 if "/proposition/" in uri else 0.0
    concept_penalty = -1.0 if "/concept/" in uri and word_count < 8 else 0.0
    return base + math.log1p(max(lucene, 0.0)) + substantive_bonus + proposition_bonus + concept_penalty


def merge_chunks(*chunk_lists: list[dict[str, Any]]) -> list[dict[str, Any]]:
    merged: dict[tuple[str | None, str | None, str | None, str | None], dict[str, Any]] = {}
    for chunks in chunk_lists:
        for chunk in chunks:
            key = (
                chunk.get("node_uri"),
                chunk.get("text"),
                chunk.get("sentence_uri"),
                chunk.get("section_uri"),
            )
            existing = merged.get(key)
            if existing is None:
                merged[key] = dict(chunk)
                continue
            existing_score = existing.get("score") or 0.0
            new_score = chunk.get("score") or 0.0
            existing["score"] = max(existing_score, new_score)
    return list(merged.values())


def fetch_paper_metadata(paper_id: str) -> list[dict[str, str | None]]:
    paper_uri = paper_uri_from_id(paper_id)
    rows = run_select(paper_metadata_query(paper_uri))
    return [
        {
            "predicate": binding_value(row, "predicate"),
            "value": binding_value(row, "value"),
            "display_value": binding_value(row, "displayValue"),
        }
        for row in rows
    ]


def fetch_all_paper_chunks(paper_id: str) -> list[dict[str, Any]]:
    paper_uri = paper_uri_from_id(paper_id)
    return rows_to_chunks(run_select(all_paper_chunks_query(paper_uri)))


def retrieve_relevant_chunks(paper_id: str, question: str, limit: int = 12) -> list[dict[str, Any]]:
    paper_uri = paper_uri_from_id(paper_id)
    indexed_chunks: list[dict[str, Any]] = []
    if lucene_query_from_question(question):
        indexed_chunks = rows_to_chunks(run_select(relevant_chunks_query(paper_uri, question, limit=max(limit * 5, 40))))

    # Pull the paper-scoped evidence too, then rerank. This keeps the text index in the loop
    # but avoids short concept labels dominating answer generation.
    chunks = merge_chunks(indexed_chunks, fetch_all_paper_chunks(paper_id))
    for chunk in chunks:
        chunk["answer_score"] = chunk_answer_score(question, chunk)
    chunks.sort(key=lambda item: item.get("answer_score") or 0.0, reverse=True)
    return chunks[:limit]


## Answer Function

This notebook uses an extractive answer by default: it returns the highest-ranked paper-grounded evidence snippets and attributes them to section URIs. You can replace `build_extractive_answer` with an LLM call later while keeping the same retrieval and attribution layer.

In [8]:
def select_answer_chunks(chunks: list[dict[str, Any]], max_chunks: int = 4) -> list[dict[str, Any]]:
    substantive = [chunk for chunk in chunks if len(str(chunk.get("text", "")).split()) >= 8]
    candidates = substantive or chunks
    selected: list[dict[str, Any]] = []
    seen_text: set[str] = set()
    for chunk in candidates:
        text = str(chunk.get("text", "")).strip()
        if not text or text in seen_text:
            continue
        seen_text.add(text)
        selected.append(chunk)
        if len(selected) >= max_chunks:
            break
    return selected


def build_extractive_answer(question: str, chunks: list[dict[str, Any]], max_chunks: int = 4) -> str:
    selected = select_answer_chunks(chunks, max_chunks=max_chunks)
    if not selected:
        return "I could not find paper-grounded evidence for this question."
    return " ".join(str(chunk["text"]).strip() for chunk in selected)


def dotenv_value(*names: str, default: str | None = None) -> str | None:
    for name in names:
        value = os.getenv(name)
        if value:
            return value

    env_path = os.getenv("ENV_FILE", ".env")
    try:
        lines = open(env_path, encoding="utf-8").read().splitlines()
    except OSError:
        return default

    wanted = set(names)
    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or "=" not in stripped:
            continue
        key, raw_value = stripped.split("=", 1)
        if key.strip() not in wanted:
            continue
        value = raw_value.strip().strip('"').strip("'")
        if value:
            return value
    return default


def format_evidence_for_llm(chunks: list[dict[str, Any]], max_chunks: int = 4) -> str:
    selected = select_answer_chunks(chunks, max_chunks=max_chunks)
    blocks: list[str] = []
    for index, chunk in enumerate(selected, start=1):
        section_uri = chunk.get("section_uri") or "unavailable"
        section_title = chunk.get("section_title") or "Untitled section"
        node_uri = chunk.get("node_uri") or "unavailable"
        text = str(chunk.get("text", "")).strip()
        blocks.append(
            f"[E{index}]\n"
            f"section_title: {section_title}\n"
            f"section_uri: {section_uri}\n"
            f"node_uri: {node_uri}\n"
            f"text: {text}"
        )
    return "\n\n".join(blocks)


def openrouter_chat_completion(
    messages: list[dict[str, str]],
    *,
    model: str | None = None,
    temperature: float = 0.1,
    max_tokens: int = 700,
    timeout: float = 60.0,
) -> str:
    api_key = dotenv_value("OPENROUTER_API_KEY", "OPEN_ROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not set in the environment or .env")

    selected_model = model or dotenv_value("OPENROUTER_MODEL", "OPENROUTER_MODEL_NAME", "OPEN_ROUTER_MODEL")
    if not selected_model:
        raise RuntimeError("OPENROUTER_MODEL is not set in the environment or .env")

    api_url = dotenv_value("OPENROUTER_API_URL", default="https://openrouter.ai/api/v1/chat/completions")
    payload = {
        "model": selected_model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": dotenv_value("OPENROUTER_SITE_URL", default="http://localhost") or "http://localhost",
        "X-Title": dotenv_value("OPENROUTER_APP_NAME", default="paper-convo-buddy") or "paper-convo-buddy",
    }
    request = Request(
        api_url,
        data=json.dumps(payload).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            body = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:1000]
        raise RuntimeError(f"OpenRouter returned HTTP {exc.code}: {detail}") from exc
    except URLError as exc:
        raise RuntimeError(f"Could not reach OpenRouter: {exc.reason}") from exc

    try:
        return body["choices"][0]["message"]["content"].strip()
    except (KeyError, IndexError, TypeError) as exc:
        raise RuntimeError(f"OpenRouter response did not contain a chat message: {body}") from exc


def generate_answer_openrouter(question: str, chunks: list[dict[str, Any]], max_chunks: int = 4) -> str:
    evidence = format_evidence_for_llm(chunks, max_chunks=max_chunks)
    if not evidence:
        return "I could not find paper-grounded evidence for this question."

    messages = [
        {
            "role": "system",
            "content": (
                "You answer questions about a paper using only the supplied evidence. "
                "Be concise, scholarly, and faithful to the evidence. "
                "Do not invent details. If the evidence is insufficient, say so. "
                "Do not include a bibliography; section URI attribution is handled separately."
            ),
        },
        {
            "role": "user",
            "content": f"Question:\n{question}\n\nEvidence:\n{evidence}\n\nAnswer:",
        },
    ]
    return openrouter_chat_completion(messages)


def attribution_uris(chunks: list[dict[str, Any]], used_limit: int = 4) -> list[str]:
    uris: list[str] = []
    for chunk in chunks[:used_limit]:
        uri = chunk.get("section_uri") or chunk.get("sentence_uri") or chunk.get("node_uri")
        if isinstance(uri, str) and uri not in uris:
            uris.append(uri)
    return uris


AnswerBuilder = Any


def answer_question(
    paper_id: str,
    question: str,
    top_k: int = 12,
    answer_chunks: int = 4,
    answer_mode: str = "openrouter",
    answer_builder: AnswerBuilder | None = None,
) -> dict[str, Any]:
    chunks = retrieve_relevant_chunks(paper_id, question, limit=top_k)
    used_chunks = select_answer_chunks(chunks, max_chunks=answer_chunks)
    if answer_builder is None:
        normalized_mode = answer_mode.strip().lower()
        if normalized_mode in {"openrouter", "generative", "llm"}:
            answer_builder = generate_answer_openrouter
        elif normalized_mode == "extractive":
            answer_builder = build_extractive_answer
        else:
            raise ValueError(f"Unknown answer_mode: {answer_mode}")
    answer = answer_builder(question, used_chunks, max_chunks=answer_chunks)
    return {
        "question": question,
        "answer": answer,
        "attribute_uris": attribution_uris(used_chunks, used_limit=answer_chunks),
        "answer_mode": answer_mode,
    }


## Try It

Change `PAPER_ID` and `QUESTION`, then run the cells below.

In [9]:
PAPER_ID = "towards_faithfully_interpretable_nlp_systems_how_should_we_define_and_evaluate_faithfulness"
QUESTION = "What exactly do the authors mean by faithfulness in NLP interpretability?"

result = answer_question(PAPER_ID, QUESTION)
result


{'question': 'What exactly do the authors mean by faithfulness in NLP interpretability?',
 'answer': "In the context of NLP interpretability, the authors define faithfulness as the property of an interpretation method that accurately represents the reasoning process behind a model's prediction. They note that faithfulness is often treated as a binary property, where an interpretation can be deemed faithful or not. However, they also express concerns about the practicality of fully satisfying the assumptions required for faithfulness, suggesting that it is easy to disprove the faithfulness of an interpretation method through counter-examples.",
 'attribute_uris': ['https://w3id.org/twc/sudo/kg/section/c030a34d-5f23-46d7-9379-45eac576ae36',
  'https://w3id.org/twc/sudo/kg/section/adc204f4-7f4d-4c28-9a9b-b95e8711fe06'],
 'answer_mode': 'openrouter'}

In [10]:
# Inspect the retrieved evidence when debugging answer quality.
retrieve_relevant_chunks(PAPER_ID, QUESTION, limit=5)


[{'node_uri': 'https://w3id.org/twc/sudo/kg/proposition/6fd7d7be-666a-4914-b658-0ffad7a143bf',
  'text': 'What does it mean for an interpretation method to be faithful?',
  'sentence_uri': 'https://w3id.org/twc/sudo/kg/sentence/5a855824-51b6-482a-9c0a-2bca72f557e7',
  'section_uri': 'https://w3id.org/twc/sudo/kg/section/c030a34d-5f23-46d7-9379-45eac576ae36',
  'section_title': 'Defining Faithfulness',
  'section_position': 8,
  'sentence_position': 1,
  'score': None,
  'node_types': ['https://w3id.org/twc/sudo/ontology#Descriptor',
   'https://w3id.org/twc/sudo/ontology#Elaboration'],
  'answer_score': 7.0},
 {'node_uri': 'https://w3id.org/twc/sudo/kg/proposition/2f76b523-8a33-4510-9ef6-61b07e4530ff',
  'text': 'Finally, we observe a trend by which faithfulness is treated as a binary property, followed by showing that an interpretation method is not faithful.',
  'sentence_uri': 'https://w3id.org/twc/sudo/kg/sentence/74cfbdb4-031d-4f4a-9988-6deb008e65eb',
  'section_uri': 'https://w3i